<a id="multi"></a>
### Multivariate Linear Regression

- QUESTION: What do think "multivariate linear regression" means?
- In ML, we're often using multiple features in our prediction
    - If we're using Linear Regression in Machine Learning, it's almost always multivariate
- Remember, calculating all the coefficients for each feature always boils down to finding the right slope or (m) in `y = mx + b`
- Let's run linear regression again but with the all the features

In [ ]:
#Suppresses a warning when loading the Boston dataset from sklearn
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

#allow us to access sklearn datasets
from sklearn import datasets

#sklearn code to load Boston dataset
boston = datasets.load_boston()
boston;

- We saved the Boston dataset into a variable called `boston`. Looking below, it contains a lot of information and it is not suitable yet for data manipulation.

In [ ]:
boston

In [ ]:
import pandas as pd
import numpy as np

#Create a dataframe with features X and the name of the features from above
df = pd.DataFrame(X, columns=boston.feature_names)

In [ ]:
df.head()

In [ ]:
#Drop the dependent variable
#Capital "X" because it has multiple features
X = df.drop(columns=['PRICE'])

#Set our dependent variable
y = df['PRICE']

In [ ]:
# Initialize ML algorithm
lr = LinearRegression()

# "Train" or fit our features (X) and price (y) to the linreg model
lr.fit(X, y)

# Predict the price and save it to a variable
prediction_all = lr.predict(X)

# Plot
fig, ax = plt.subplots()
ax.scatter(y, prediction_all, edgecolors=(0, 0, 0))
ax.plot([y.min(), y.max()], 
        [y.min(), y.max()], 
        'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

# Calculate MSE
mse_all = mean_squared_error(y, prediction_all)
print('The mean squared error (MSE) is %f' % mse_all)

In [ ]:
#Saving this for when we talk about scaling. Ignore for now.
df_coef = pd.DataFrame({'Column': X.columns, 'coefficient': lr.coef_})

- Our new MSE of 21.9 is almost half of our univariate linreg model and 1/4 of our original benchmark MSE of 84.2!
- What can we do to reduce our MSE?
    - When would it NOT be good to use all our features in prediction?
        - What if we had errors in one of the columns?
        - What if we had a column that tracked the amount of dates an anymous data scientist had per year?
            - Would this particular feature intuitively help in the prediciton of the price of homes?
- Let's try an experiment:

<a id="random"></a>
### Random features
- Let's perform linear regression with just the three features: Age, Tax and RM
- Why did we choose this? Random guess, no other

In [ ]:
#We can look at our full dataset again
df.head()

In [ ]:
#Save just the three features
X = df[["AGE","TAX","RM"]]

#Set y
y = df['PRICE']

# Initialize ML algorithm
lr = LinearRegression()

# "Train" or fit our three features (X) and price (y) to the linreg model
lr.fit(X, y)

# Predict the price and save it to a variable
prediction_three = lr.predict(X)

# Plot
fig, ax = plt.subplots()
ax.scatter(y, prediction_three, edgecolors=(0, 0, 0))
ax.plot([y.min(), y.max()], 
        [y.min(), y.max()], 
        'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

# Calculate MSE
mse_three = mean_squared_error(y, prediction_three)
print('The mean squared error (MSE) is %f' % mse_three)

- The model performed almost as poorly as just using the "RM" feature by itself
- Is there a way to find the best features to use in a model?

<a id="feat_sel"></a>

## Feature Selection: Intro to Correlation

- Numerous methods to perform feature selection on a dataset exist
    - They will be covered in the rest of the notebook
- For now, let's focus on **Correlation**
- Correlation measures the strength of the linear relationship between two variables/attributes/features
- A typical example is height and weight (or say, income and taxes paid). Those two variables are often correlated. Why is this so?

<figure>
  <img src='https://www.simplypsychology.org/correlation.jpg'/>
  <figcaption>Image from <a href='https://www.simplypsychology.org/correlation.html'>SimplyPsychology: Correlation</a></figcaption>
</figure>

**What does this mean for Machine Learning?**

- Correlation is **good** for a feature (x) used to predict a target variable (y)
- However, there can be issues if there is correlation BETWEEN features.  
- This phenomenon is known as **Multicollinearity**
    - When the two independent variables are *highly* correlated that there becomes a linear association between the two
- For e.g. consider a small set of features used to measure university acceptance like
    1. Student's overall grades
    2. GPA
    3. SAT Score
    4. Extracurricular activities
- GPA has a high correlation with the student's overall grades. 
    - So much so in fact that if the grades flunctuate in any way, it could be seen that the GPA would fluctuate with it
    - Fluctuations in grade → same fluctuation in GPA → chances of an amplified fluctuation in predicted variable
    - What we want: independent signals!  
- Sometimes, accuracy may not be affected. But, the results might not be **interpretabile** is important.
- Interpretability just means the ability to explain the "ins-and-outs" of how you came to your prediction.
    - Some algorithms are very difficult to explain to stakeholders.
    - We'll see examples of these during the course

<a id="heatmap"></a>
### Heatmap
- We can determine the correlation between a target variable and features 
- and also if there is any kind of relationship between feature variables. 

In [ ]:
#.corr is a Pandas method to find pairwise correlation
df.corr().shape

In [ ]:
#set the graph style
sns.set(style="white")

# Compute the correlation matrix
corr = df.corr()

# Generate a mask for the upper triangle
mask = np.zeros_like(corr, dtype=bool) # Return an array of zeros with the same shape and type as a given array
mask[np.triu_indices_from(mask)] = True # Return the indices for the upper-triangle of arr

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True)

- In the cell above, the mask is used to hide the upper triangle of the heat map. 
    - This is because it exactly mirrors the bottom triangle
- Correlation strength ranges from -1 to +1
    - More **red** the square is, the more positively correlated the two variables are
    - More **blue** the square is, the more negatively correlated the variables are
    - QUESTION: What are some examples of **multicollinearity?**    
- Just because there *seems* to be a correlation between variables it does not necessarily mean that one exists. 
    - This should be taken into account when selecting features.
- Also, this heatmap only accounts for numerical data.
    - Categorical features such as "Spam or Not Spam" will be ignored
- Feature Selection: Find the bottom row as it represents correlations with our Price variable.
    - Let's choose any feature greater with correlation strength greater than or equal to (+/-) 0.30
    - Let's see if this improves our prediction

In [ ]:
#peek at our dataset
df.head()

In [ ]:
#Save the features with correlation >= (+/-) 0.3
X = df.drop(columns=["PRICE", "CHAS","DIS"])
X.head()

In [ ]:
#Set our dependent variable
y = df['PRICE']

# Initialize ML algorithm
lr = LinearRegression()

# "Train" or fit our three features (X) and price (y) to the linreg model
lr.fit(X, y)

# Predict the price and save it to a variable
prediction_ptthree = lr.predict(X)

# Plot
fig, ax = plt.subplots()
ax.scatter(y, prediction_ptthree, edgecolors=(0, 0, 0))
ax.plot([y.min(), y.max()], 
        [y.min(), y.max()], 
        'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

# calculate MSE
mse_ptthree = mean_squared_error(y, prediction_ptthree)
print('The mean squared error (MSE) is %f' % mse_ptthree)

- The MSE is lower than the three random features we chose, but still performed a little worse than just using all the features. 
    - Choosing the correlation cutoff of (+/-) 0.30 was based on our discretion
    - The heatmap is helpful but it's up to us how to use it


<a id="ex1"></a>
#### Exercise 1 

- Try combinations of features and see if you can get a lower MSE        
- What else can we do to better our prediction?

In [ ]:
# YOUR CODE HERE



<a id="scale"></a>
## Scaling

- Look at the Boston dataset below:
- What can you tell about the ranges of numbers between the different features
- For example, let's compare the values of "CRIM" to the values of "B"

In [ ]:
df.head()

- **QUESTION**: What could potentially be the issue with this large difference in values?

<figure>
  <img src='https://miro.medium.com/max/1400/1*yR54MSI1jjnf2QeGtt57PA.png' width='50%'/>
  <figcaption>Image from <a href='https://towardsdatascience.com/all-about-feature-scaling-bcc0ad75cb35' style='width:500px'>TowardsDataScience: All About Feature Scaling</a></figcaption>
</figure>

- ML modelling → sees it as a number/geometry poroblem
    - Units becomes an issue → what if one feature is say, distance (km) and we convert it into nm? Or price in (millions) to cents?
- Can result in additional focus to such features regardless of whether they demand such focus
- What we want is for the model to give each feature a fair chance
    - Scaling the data → removes any externalities we bring to the modelling process via arbitrary choice of units
- Also, Not all ML algorithms need to have their data scaled, but it's a good rule of thumb
- Many types of scaling, available in sklearn, of which 2 most common ones are:
    - **Standardization**
    - **Min-Max Scaling**
- Standardization/Z-Score (sklearn: `StandardScaler`)
    - It rescales the features to have the properties for a standard normal distribution
        - mean of 0 and standard deviation of 1
    - Works quite well if data is close to being normally distributed
- Min-Max Scaling (sklearn: `MinMaxScaler`)
    - Changes range of the data to be between 0 and 1
    - Min is pegged to 0 and max is pegged to 1, basicallyy
    - Useful when data is not normally distributed

<a id="stand"></a>
### StandardScaler

In [ ]:
# Look at our data
df.head()

In [ ]:
# 1. Import StandardScaler from sklearn
from sklearn.preprocessing import StandardScaler

# 2. Initialize 
scaler = StandardScaler()

# 3. Fit_transform our data
# Note: we don't need to scale our dependent variable
data_scaled = scaler.fit_transform(df.drop(columns=['PRICE']))


- The method "fit_transform()" is actually a combination of two methods
    - .fit(): Fits the data and learns the parameters
    - .transform(): Uses those same parameters to transform the data
- We used the shortcut fit_transform() but you'll see an instance of doing them separately later
- The .fit() and .transform() method examples are below which are equivalent to what we did above:
        
```
# 3. Fit to our data
scale = scaler.fit(df.drop(columns=['PRICE']))

# 4. Transform our fitted data
data_scaled = scale.transform(df.drop(columns=['Price']))
```

In [ ]:
#Here's what our scaled features look like now
df_features_scaled = pd.DataFrame(data_scaled, columns=boston.feature_names)
df_features_scaled.head()

In [ ]:
# set X to our scaled features
X = df_features_scaled

# set y
y = df['PRICE']

# Initialize ML algorithm
lr = LinearRegression()

# "Train" or fit our scaled features features (X) and price (y) to the linreg model
lr.fit(X, y)

# Predict the price and save it to a variable
prediction_scaled = lr.predict(X)

# Plot
fig, ax = plt.subplots()
ax.scatter(y, prediction_scaled, edgecolors=(0, 0, 0))
ax.plot([y.min(), y.max()], 
        [y.min(), y.max()], 
        'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

# compute MSE
mse_scaled = mean_squared_error(y, prediction_scaled)
print('The mean squared error (MSE) is %f' % mse_scaled)

- Notice how scaling our features didn't actually change our MSE. It usually won't for Linear Regression, but it is vital for comparing the **model coefficients**. We can access these coefficients using the `.coef_` attribute:

In [ ]:
lr.coef_

In [ ]:
df_coef_scaled = pd.DataFrame({'Column': X.columns, 'coefficient': lr.coef_})
df_coef_scaled

- The larger the absolute value of the coefficient, the greater that specific feature affects the model. That being said, we can also perform **Feature Selection** by choosing the features with the largest coefficients. Which features would you choose based on its coefficients?
- **Question**: Why did we have to scale the features?
- Take a look at the **unscaled** feature coefficients below

In [ ]:
df_coef

- Would you still pick the same features?

#### Exercise 2

- Using the scaled model coefficients, choose the "most important" features for your own model. What's the MSE?

In [ ]:
# YOUR CODE HERE



<a id="leak"></a>
## Data Leakage

- Let's revisit Bias and Variance
- QUESTION: What's wrong with training our models on ALL of our data? 

### train_test_split

- ML "learns" from the data it is exposed to 
    - No matter the algorithm, if it data is exposed to it (during fitting, transformation etc), it will be used
    - Control the exposure → ultimately, we want a ML model that performs in the real world, on data that it is yet to see
- Often times, we have one single big dataset to use for models
- Divide them into "training" and a "test" set. 
- Model only "sees" the training data when it learns → i.e., it is not exposed to the test data
  - Prevents data leakage
- "Test" set is only used to finally evaluate the model

<figure>
  <img src='https://miro.medium.com/max/948/1*4G__SV580CxFj78o9yUXuQ.png' width='50%'/>
</figure>

- Let's use it with our dataset:

In [ ]:
#import train_test_split
from sklearn.model_selection import train_test_split

#denote our features. We'll use the unscaled features from our original DataFrame
X = df.drop(columns=['PRICE']) 
X.head()

- Below is the function for train_test_split()<br><br>

In [ ]:
#Splitting our dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

- Inside the function we have four arguments:
    - Our feature variables (X)
    - Our dependent variable (y)
    - test_size (0.33) which means how much of the dataset will be split into the test set
        - 1/3 for test set and 2/3 for training data set
    - random_state (42) is any number used to reproduce the same results. 
        - It remembers the random split of our data.
- Look at the **left** side of the equation:
    - We have our training features set (X_train)
    - Testing features set (X_test)
    - Training target set (y_train)
    - Testing target set (y_test)

What happens after using `train_test_split`?
- Original data is split with 67% as data for training and 33% to be tested
- Can verify this by looking at the shapes

In [ ]:
# Verifies if condition is true
assert X_train.shape[0] + X_test.shape[0] == X.shape[0]
X_train.shape, y_train.shape, X_test.shape, y_test.shape

- Let's use our split dataset for prediction:

In [ ]:
# Initialize ML algorithm
lr = LinearRegression()

# Train our model with our training set
lr.fit(X_train, y_train) #Formely (X, y)

# Predict the price and save it to a variable
y_pred = lr.predict(X_test) #Formely X

# Plot
fig, ax = plt.subplots()
ax.scatter(y_test, y_pred, edgecolors=(0, 0, 0))
ax.plot([y_test.min(), y_test.max()], 
        [y_test.min(), y_test.max()], 
        'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

# compute MSE
mse_split = mean_squared_error(y_test, y_pred)
print('The mean squared error (MSE) is %f' % mse_split)

- The MSE is very similar to our MSE trained on our full dataset.
- However, our model is now more **robust** and should perform better with new or unseen data we collect

<a id="together"></a>

## Putting it all together

- We've covered
    1. linear regression
    2. feature selection
    3. scaling
    4. splitting the data
- Thankfully, what we have learned is often repeated when modeling our data
    - You'll become comfortable with practice!
- For now, let's put it all together:
- Let's perform Linear Regression with what we learned
- Assume we performed our Heatmap and have chosen our features to use in our prediction
- We'll include scaling and train_test_split

In [ ]:
#peek at our dataset
df.head()

In [ ]:
#Save the features with correlation >= (+/-) 0.3
X = df.drop(columns=["PRICE", "CHAS","DIS"])
X.head()

- We split and then scale our data to prevent data leakage
- Method adapted from this [article](https://datascience.stackexchange.com/questions/54908/data-normalization-before-or-after-train-test-split).

In [ ]:
# Split our data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Initialize Scaler 
scaler = StandardScaler()

# fit and transform our training features
# Remember, we don't need to scale our dependent variable
X_train_scaled = scaler.fit_transform(X_train)

# Initialize ML algorithm
lr = LinearRegression()

# Fit our data to the model
lr.fit(X_train_scaled, y_train)

# Scale our test features based on the parameters learned from our training features
# Note the difference of .transform() method instead of .fit_transform() like before
X_test_scaled = scaler.transform(X_test)

# Predict the price and save it to a variable
y_pred = lr.predict(X_test_scaled)

# Plot
fig, ax = plt.subplots()
ax.scatter(y_test, y_pred, edgecolors=(0, 0, 0))
ax.plot([y_test.min(), y_test.max()], 
        [y_test.min(), y_test.max()], 
        'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

# compute MSE
mse_fin = mean_squared_error(y_test, y_pred)
print('The mean squared error (MSE) is %f' % mse_fin)

Congratulations! You just performed your first complete model that includes preprocessing

<a id="summary"></a>

## Summary of Learning Outcomes

1. [Intro to Sci-Kit Learn](#sci)
    - Sci-Kit Learn (Sklearn) is the premier Python library for preprocessing and modeling data
    - Throughout this lesson, we used the [Boston Dataset](#bost)
        - We explored it and looked at some Sklearn specific code in loading and manipulating it
<br><br>
2. [Linear Regression](#linreg)
    - It is a simple that predicts a dependent variable from a set of independent variables
        - Great to use as a basline
        - Advanced linear algorithms built from linear regression
        - In [Dreaded Formula](#formula) we looked at the formula for linear regression
            - Just remember that it boils back down to `y = mx + b`
            - "Least Squares Criterion" is how we create the regression line and it aims to reduce the errors between our predictions and our actual "y" values
    - In [Dependent Variable Only](#dep_only), we explored what happens when we try to predict the price of homes when we only have the price of homes
    - In [Evaluation Metric](#eval), we explained why we have them and some common ones for linear regression
        - r squared is a popular metric that ranges from 0 to (+/-) 1 with (+/-) 1 being the best
        - MSE measures the error between our predicted values and our actual "y" values. 
            - We used it to judge our models in this lesson 
    - [Simple Linear Regression](#simp) has one independent variable to predict a dependent one
    - In [Define the Model](#define), we walked through an example of performing simple linear regression with Sklearn
    - In [Multivariate Linear Regression](#multi), we performed linear regression but with all our features
    - In [Random Features](#random), we performed linear regression with randomly chosen features
        - The goal was to ask ourselves how to pick the best features
<br><br>
3. [Intro to Feature Selection](#feat_sel)
    - We spoke about correlation which shows a linear relationship between two variables `df.corr()`
        - It is good when we see correlation between an independent variable and a dependent variable
        - It is not so good when we see it between independent variables
            - This could hurt data integrity as well as model interpretability 
    - [Heatmap](#heatmap) is a visual which shows potential correlation between variables.
        - We used it to choose our features
<br><br>  
4. [Scaling](#scale)
    - Scaling puts all our features on the same range
        - Sklearn has many functions to scale our features but we explored two common ones:
            - Standardization transforms our data on a scale centered around 0.`from sklearn.preprocessing import StandardScalar`
                - It has the assumption our data is normally distributed
            - Normalization transforms our data on a scale form 0 to 1`from sklearn.preprocessing import MinMaxScalar`
                - It does not make assumptions on the distribution of our data
    - In [StandardScaler](#stand), we used Sklearn to "Standardize" our data before performing linear regression
        - We can use the scaled model coefficients for Feature Selection
<br><br>
5. In [Data Leakage](#leak), we learned that training our models on all our data introduces bias.
    - This hurts our prediction as it will not perform as well on new data
    - ML algorithms learn on any data it "sees" so we have to "hide" some data from it. 
        - We use this "hidden" (or unseen) data to test the trained model
    - `train_test_split` is the sklearn function we used split our data into training and testing data
        - train data is "seen" by the model
        - test data is "hidden" and only used to evaluate it once it's trained
<br><br>
6. [Putting it all Together](#together)
    - We combined all our preprocessing and modeling into one section
        1. We subset our dataset so we only kept the ones that "good" correlation with our dependent variable
        2. We split our data using train_test_split
        3. We scaled our training features
        4. We fit our data to the linear regression algorithm
        5. We scaled the test features
        6. We made our predictions
        7. We plotted and calculated our MSE
    - Many of these steps will be repeated throughout the course

---

<a id="ref"></a>
## Additional Information

- Boston Dataset Examples:
    1. [Neural.cz](http://www.neural.cz/dataset-exploration-boston-house-pricing.html)
    2. [Medium.com @Haydar_ai](https://medium.com/@haydar_ai/learning-data-science-day-9-linear-regression-on-boston-housing-dataset-cd62a80775ef)
    3. [University Toronto](https://www.cs.toronto.edu/~delve/data/boston/bostonDetail.html)
- SciKit Learn Information:
    1. [MachineLearningMaster: Intro to SciKit Learn](https://machinelearningmastery.com/a-gentle-introduction-to-scikit-learn-a-python-machine-learning-library/)
    2. [Scikit Learn Basic Tutorial](http://scikit-learn.org/stable/tutorial/basic/tutorial.html) 
    3. [Linear Regression in SciKit Learn](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression)
    4. [Auto Example in SciKit learn](http://scikit-learn.org/stable/auto_examples/plot_cv_predict.html#sphx-glr-auto-examples-plot-cv-predict-py)
- Linear Regression Information:
    1. [Biostat Handbook](http://www.biostathandbook.com/linearregression.html)
    2. [LearnStatisticswithR](https://learningstatisticswithr.com/book/regression.html#introregression)
    3. [MSE or R-Squared: which one to use](https://vitalflux.com/mean-square-error-r-squared-which-one-to-use/)
- Pre-processing (scaling, feature selection, splitting):
    1. [Normalization before or after train_test_split](https://datascience.stackexchange.com/questions/54908/data-normalization-before-or-after-train-test-splithttp://localhost:8888/lab/tree/upskill/04%20-%20Intro%20to%20Modelling/02%20-%20Linear%20Regression-Copy1.ipynb)
    2. [Difference between Scikit Learn .fit() and .transform() methods](https://www.analyticsvidhya.com/blog/2021/04/difference-between-fit-transform-fit_transform-methods-in-scikit-learn-with-python-code/)
    3. [SciKit Learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
    4. [Need and Types of Feature Scaling](https://medium.com/analytics-vidhya/need-and-types-of-feature-scaling-101ac2fc3af0)
    5. [What is Multicollinearity?](https://www.analyticsvidhya.com/blog/2020/03/what-is-multicollinearity/)
- [Machine Learning: Testing and Error Metrics](https://www.youtube.com/watch?v=aDW44NPhNw0&list=PLs8w1Cdi-zvYKddqF_TFR7_yXuinufEPN&index=3) gives you a good overview of various metrics. The same channel have a specific video on [ROC](https://www.youtube.com/watch?v=z5qA9qZMyw0&list=PLs8w1Cdi-zvYKddqF_TFR7_yXuinufEPN&index=3)
- Check out [this friendly introduction to Linear Regression](https://www.youtube.com/watch?v=wYPUhge9w5c&list=PLs8w1Cdi-zvYKddqF_TFR7_yXuinufEPN&index=4)